# 08 Reproducibility Artifact Audit

Verifies the full-mode stage 01-07 evidence stored in Google Drive, records file hashes and row counts, and captures the runtime environment. Raw datasets are not copied.

In [2]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys
import time

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
EXPECTED_DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = EXPECTED_DRIVE_PROJECT_ROOT

def is_project_root(path):
    return (path / 'configs' / 'project_colab.json').is_file() and (path / 'src' / 'ecotrust').is_dir()

def discover_mounted_project():
    candidates = [EXPECTED_DRIVE_PROJECT_ROOT]
    my_drive = Path('/content/drive/MyDrive')
    candidates.extend([
        my_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
        my_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
    ])
    shared_drives = Path('/content/drive/Shareddrives')
    if shared_drives.is_dir():
        for shared_drive in shared_drives.iterdir():
            candidates.extend([
                
                shared_drive / 'projects' / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
                shared_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
                shared_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION',
            ])
    return next((candidate for candidate in candidates if is_project_root(candidate)), None)

try:
    from google.colab import drive  # type: ignore
    if not is_project_root(EXPECTED_DRIVE_PROJECT_ROOT):
        print('Drive project is not visible; resetting stale DriveFS state...', flush=True)
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        subprocess.run(
            ['fusermount', '-uz', '/content/drive'],
            check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        shutil.rmtree('/content/drive', ignore_errors=True)
        Path('/content/drive').mkdir(parents=True, exist_ok=True)
        for mount_attempt in range(1, 4):
            try:
                drive.mount('/content/drive', force_remount=False, timeout_ms=120000)
                break
            except Exception as mount_exc:
                if mount_attempt == 3:
                    raise
                wait_seconds = 15 * mount_attempt
                print(
                    f'Drive mount attempt {mount_attempt}/3 failed: '
                    f'{type(mount_exc).__name__}: {mount_exc}. '
                    f'Retrying in {wait_seconds}s...',
                    flush=True,
                )
                time.sleep(wait_seconds)
    discovered_root = discover_mounted_project()
    if discovered_root is not None:
        DRIVE_PROJECT_ROOT = discovered_root
except ImportError:
    pass
except Exception as exc:
    raise RuntimeError(f'Google Drive project mount failed: {type(exc).__name__}: {exc}') from exc

def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError('Cannot find the EcoTrust-FLIDS IMPLEMENTATION folder in the mounted Google Drive.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
CONFIG_PATH = PROJECT_ROOT / 'configs' / ('project_colab.json' if str(PROJECT_ROOT).startswith('/content/drive') else 'project.json')
RUN_MODE = 'full'
DEVICE = 'cpu'
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']

importlib.invalidate_caches()
import ecotrust.evaluation.artifact_audit as audit_module
importlib.reload(audit_module)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)
print('RUN_MODE:', RUN_MODE)
print('DEVICE:', DEVICE)
print('AUDIT_SOURCE:', audit_module.__file__)

result = audit_module.run_reproducibility_audit(
    CONFIG_PATH,
    run_mode=RUN_MODE,
    datasets=DATASETS,
)
print('STAGE_08_STATUS:', result['status'])
print('STAGE_08_MANIFEST:', result['manifest'])
print('ARTIFACT_COUNT:', result['artifact_count'])


PROJECT_ROOT: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION
CONFIG_PATH: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/configs/project_colab.json
RUN_MODE: full
DEVICE: cpu
AUDIT_SOURCE: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/src/ecotrust/evaluation/artifact_audit.py
STAGE_08_STATUS: completed
STAGE_08_MANIFEST: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/logs/08_reproducibility_audit_full_manifest.json
ARTIFACT_COUNT: 177
